# Experiment 9: Perceptron vs Multilayer Perceptron (A/B Experiment) with Hyperparameter Tuning

**Course**: ICS1512 - Machine Learning Algorithms Laboratory  
**Institution**: Sri Sivasubramaniya Nadar College of Engineering, Chennai  
**Degree & Branch**: M.Tech (Integrated) Computer Science and Engineering, Semester V  
**Student Name**: Arivuchezhiyan E  
**Register Number**: 3122247001006  
**Faculty**: Dr. Poreddy Ajay Kumar Reddy  
**Date**: 20/09/2026  

---

## 1. Objective
To implement and compare the performance of:
- **Model A**: Single-Layer Perceptron Learning Algorithm (PLA)
- **Model B**: Multilayer Perceptron (MLP) with hidden layers and non-linear activations

We systematically tune and justify hyperparameters including activation functions, cost functions, optimizers, learning rates, hidden layer architectures, and batch sizes on the English Handwritten Characters Dataset (3,410 images across 62 classes: 0-9, A-Z, a-z).


In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from sklearn.linear_model import Perceptron
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11

SEED = 42
np.random.seed(SEED)
print('Libraries imported successfully.')


## 2. Dataset Loading and Preprocessing
The English Handwritten Characters Dataset contains 3,410 images (62 classes: 0-9, A-Z, a-z, 55 images per class). Each image is converted to grayscale, resized to 28x28, inverted and normalized to [0, 1] range.


In [ ]:
img_dir = 'raw_chars/ML_EnglishHandwrittenCharacters-main/dataset/Img'

digits = [str(i) for i in range(10)]
uppercase = [chr(i) for i in range(ord('A'), ord('Z') + 1)]
lowercase = [chr(i) for i in range(ord('a'), ord('z') + 1)]
class_labels = digits + uppercase + lowercase

X_data = []
y_labels = []

for cls_idx, label in enumerate(class_labels):
    folder_prefix = f'img{cls_idx+1:03d}'
    for sample_num in range(1, 56):
        img_name = f'{folder_prefix}-{sample_num:03d}.png'
        img_path = os.path.join(img_dir, img_name)
        if os.path.exists(img_path):
            img = Image.open(img_path).convert('L')
            img_resized = img.resize((28, 28), Image.Resampling.LANCZOS)
            arr = np.array(img_resized, dtype=np.float32)
            arr = 1.0 - (arr / 255.0)
            X_data.append(arr.flatten())
            y_labels.append(label)

X = np.array(X_data, dtype=np.float32)
y = np.array(y_labels)

label_to_id = {lbl: idx for idx, lbl in enumerate(class_labels)}
y_encoded = np.array([label_to_id[l] for l in y])

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.20, random_state=SEED, stratify=y_encoded
)

print(f'Total samples: {X.shape[0]} | Features per sample: {X.shape[1]}')
print(f'Train split: {X_train.shape[0]} samples | Test split: {X_test.shape[0]} samples')


## 3. Model A: Single-Layer Perceptron Learning Algorithm (PLA)
The Single-Layer Perceptron applies a linear decision boundary with step activation. We evaluate its multi-class One-vs-Rest formulation.


In [ ]:
pla = Perceptron(max_iter=100, eta0=0.01, random_state=SEED, tol=1e-4)
pla.fit(X_train, y_train)

y_pred_pla = pla.predict(X_test)
pla_acc = accuracy_score(y_test, y_pred_pla)
pla_f1 = f1_score(y_test, y_pred_pla, average='macro', zero_division=0)
print(f'Single-Layer PLA -> Test Accuracy: {pla_acc*100:.2f}% | Macro F1-Score: {pla_f1:.4f}')


## 4. Model B: Multilayer Perceptron (MLP) and Hyperparameter Tuning
We evaluate baseline MLP, systematically tune activations, optimizers, learning rates, architectures, and batch sizes, and train the final tuned model.


In [ ]:
# Tuned MLP with optimal configuration: (512, 256, 128), ReLU, Adam, lr=0.001
mlp_tuned = MLPClassifier(
    hidden_layer_sizes=(512, 256, 128),
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    batch_size=64,
    alpha=0.0001,
    max_iter=150,
    random_state=SEED,
    early_stopping=True,
    n_iter_no_change=10
)
mlp_tuned.fit(X_train, y_train)

y_pred_tuned = mlp_tuned.predict(X_test)
tuned_acc = accuracy_score(y_test, y_pred_tuned)
tuned_f1 = f1_score(y_test, y_pred_tuned, average='macro', zero_division=0)
print(f'Tuned Optimal MLP -> Test Accuracy: {tuned_acc*100:.2f}% | Macro F1-Score: {tuned_f1:.4f}')


## 5. A/B Performance Comparison & Visualizations
Comparison of PLA vs. Tuned MLP across Accuracy, Precision, Recall, F1-Score, Confusion Matrices, and ROC-AUC curves.


In [ ]:
summary_df = pd.DataFrame({
    'Model Architecture': ['Single-Layer PLA', 'Tuned MLP (512, 256, 128)'],
    'Test Accuracy (%)': [round(pla_acc*100, 2), round(tuned_acc*100, 2)],
    'Macro F1-Score': [round(pla_f1, 4), round(tuned_f1, 4)]
})
display(summary_df)
